# Evaluating the GAVEL Cognitive Element Classifier

This notebook demonstrates how to evaluate a trained GAVEL classifier on new dialogues. The evaluation process includes:

1. **Calibration**: Find optimal detection thresholds using a calibration dataset
2. **Logits Extraction**: Extract classifier logits from dialogues
3. **Evaluation**: Compute TPR, FPR, and AUC metrics on test dialogues

## Evaluation Pipeline Overview

1. **Load Models**: Load the trained RNN classifier and base LLM
2. **Extract Calibration Logits**: Process the full calibration set to extract logits
3. **Calibrate Thresholds**: Find optimal thresholds using Youden's J-statistic
4. **Extract Test Logits**: Process a few sample dialogues (positive and negative) for demo
5. **Evaluate**: Compute detection metrics on the test samples

## Prerequisites

- Trained GAVEL model (from `train_classifier.ipynb`)
- Calibration dataset (for threshold optimization)
- Test dataset with positive and negative samples
- GPU recommended for faster logits extraction

## 1. Imports and Setup

In [ ]:
import json
import os

import torch

# GAVEL imports
from gavel.config import load_config
from gavel.evaluation.calibration import calibrate
from gavel.evaluation.metrics import evaluate
from gavel.models import load_trained_classifier
from gavel.preprocessing import extract_dialogues_in_memory
from gavel.training import load_model_and_tokenizer

# Disable tokenizer parallelism warnings
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

/home/rahul/GAVEL/attention_based_classification/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
GPU: NVIDIA RTX 6000 Ada Generation
Memory: 51.0 GB


## 2. Configuration

Load the configuration file. Key evaluation parameters:
- `paths.eval_dataset`: Path to evaluation dataset (for test samples)
- `paths.calibration_dataset`: Path to calibration dataset (for threshold optimization)
- `eval.window_size`: Window size for sequence processing
- `eval.window_stride`: Stride for sliding windows

In [2]:
# Load configuration
config = load_config("notebook_config.json")
# Display key configuration values
print("=== Evaluation Configuration ===")
print(f"Base LLM: {config.model.name_or_path}")
print(f"Selected layers: {list(config.model.selected_layers)}")
print("\nEvaluation paths:")
print(f"  Eval dataset: {config.paths.eval_dataset}")
print(f"  Calibration dataset: {config.paths.calibration_dataset}")
print(f"  Calibration results: {config.paths.calibration_dir}")
print(f"  Evaluation results: {config.paths.results_dir}")
print("\nWindow parameters:")
print(f"  Window size: {config.eval.window_size}")
print(f"  Window stride: {config.eval.window_stride}")
print("\nModel path:")
print(f"  RNN: {config.paths.rnn_model_path}")

=== Evaluation Configuration ===
Base LLM: /data/thought_elements/llms/mistralai_Mistral-7B-Instruct-v0.2
Selected layers: [13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26]

Evaluation paths:
  Eval dataset: ../data/eval
  Calibration dataset: ../data/calibration
  Calibration results: ../models/mistralai_Mistral-7B-Instruct-v0.2/calibration_results
  Evaluation results: ../models/mistralai_Mistral-7B-Instruct-v0.2/results

Window parameters:
  Window size: 5
  Window stride: 5

Model path:
  RNN: ../models/mistralai_Mistral-7B-Instruct-v0.2/model/trained_model_rnn.pth


## 3. Load Models

Load both the base LLM (for attention extraction) and the trained RNN classifier.

The `gavel.models` module provides utilities for loading trained models:
- `load_trained_classifier()`: Load a trained TopicRNN from a checkpoint
- `TopicRNN`: The RNN classifier architecture

You can load your own trained models by specifying the model path and architecture parameters.

In [ ]:
from gavel.models import load_trained_classifier, TopicRNN

# Load base LLM and tokenizer
print("Loading base LLM and tokenizer...")
model, tokenizer = load_model_and_tokenizer(config.model.name_or_path)

# Load trained RNN classifier using gavel.models
# Option 1: Load from config-specified path (recommended)
# Note: config.paths.rnn_model_path already includes model_name in base_dir
rnn_model_path = config.paths.rnn_model_path

# Option 2: Load your own trained model (uncomment and modify as needed)
# rnn_model_path = "/path/to/your/trained_model_rnn.pth"

print(f"\nLoading trained RNN classifier from: {rnn_model_path}")

# Method 1: Using RNNConfig (recommended - automatically uses correct architecture)
rnn_model = load_trained_classifier(
    model_path=rnn_model_path,
    num_topics=config.num_labels,
    selected_layers_length=len(config.model.selected_layers),
    rnn_config=config.rnn,
)

# Method 2: Manual parameter specification (if you don't have RNNConfig)
# Uncomment and modify if needed:
# rnn_model = load_trained_classifier(
#     model_path=rnn_model_path,
#     num_topics=config.num_labels,
#     selected_layers_length=len(config.model.selected_layers),
#     input_dim=4096,  # Adjust based on your model (readout_dim = n_heads * head_dim)
#     hidden_dim=256,
#     num_rnn_layers=3,
#     rnn_type="GRU",
# )

# Method 3: Load manually using TopicRNN (for custom architectures)
# Uncomment if you need full control:
# rnn_model = TopicRNN(
#     num_layers=len(config.model.selected_layers),
#     input_dim=4096,  # Adjust based on your model
#     hidden_dim=config.rnn.hidden_dim,
#     num_rnn_layers=config.rnn.num_rnn_layers,
#     num_topics=config.num_labels,
#     rnn_type=config.rnn.rnn_type,
#     proj_dim=config.rnn.proj_dim,
# )
# rnn_model.load_state_dict(torch.load(rnn_model_path))
# rnn_model.eval()

print("✓ Models loaded successfully!")
print(f"  RNN model: {type(rnn_model).__name__}")
print(f"  Number of topics: {config.num_labels}")
print(f"  Selected layers: {len(config.model.selected_layers)}")

Loading base LLM and tokenizer...


Loading checkpoint shards: 100%|██████████| 3/3 [00:02<00:00,  1.08it/s]



Loading trained RNN classifier from: ../models/mistralai_Mistral-7B-Instruct-v0.2/model/trained_model_rnn.pth
✓ Models loaded successfully!
  RNN model: TopicRNN
  Number of topics: 23
  Selected layers: 14


## 4. Extract Calibration Dialogues

First, we extract and process dialogues from the **full calibration set** in-memory. This will be used to find optimal detection thresholds.

The calibration dataset is now in a separate directory (`paths.calibration_dataset`). It should contain dialogues labeled with cognitive elements, organized into `usecase_level` and `CE_level` splits. We process all of them in-memory to get comprehensive threshold estimates.

**Note**: We use `extract_dialogues_in_memory()` which processes dialogues without saving logits to disk, making the pipeline faster and more memory-efficient.

In [4]:
# Extract calibration dialogues
print("=== Step 1: Extract Calibration Dialogues ===")
print("Processing full calibration set in-memory...")
print(f"  Dataset: {config.paths.calibration_dataset}")
print(f"  Window size: {config.eval.window_size}, stride: {config.eval.window_stride}")
print("  (Processing all samples - this may take a few minutes)")


calibration_dialogues = extract_dialogues_in_memory(
    dataset_root=config.paths.calibration_dataset,
    splits=["usecase_level", "CE_level"],
    model=model,
    rnn_model=rnn_model,
    tokenizer=tokenizer,
    selected_layers=config.model.selected_layers,
    window_size=config.eval.window_size,
    window_stride=config.eval.window_stride,
    batch_size=32,
    max_samples_per_usecase=None,  # Use all calibration data
    logger=None,
    show_progress=True,
)

print(f"✓ Calibration dialogues extracted! ({len(calibration_dialogues)} dialogues)")

=== Step 1: Extract Calibration Dialogues ===
Processing full calibration set in-memory...
  Dataset: ../data/calibration
  Window size: 5, stride: 5
  (Processing all samples - this may take a few minutes)


Extracting CE_level: 100%|██████████| 23/23 [00:28<00:00,  1.25s/it]

✓ Calibration dialogues extracted! (844 dialogues)


## 5. Calibrate Thresholds

Using the calibration logits, we find optimal thresholds for each cognitive element using **Youden's J-statistic** (maximizes TPR - FPR).

The calibration process:
1. Loads all calibration dialogues
2. Sweeps over threshold values (default: 0.05 to 1.0 in 0.05 steps)
3. For each threshold, computes TPR and FPR
4. Selects the threshold that maximizes Youden's J = TPR - FPR
5. Optionally tests different patience values (how many consecutive detections needed)

In [ ]:
# Load unified ruleset
print("Loading unified ruleset...")
if config.paths.ruleset_path:
    unified_ruleset_path = config.paths.ruleset_path
else:
    # Fallback: construct path relative to base_dir
    unified_ruleset_path = os.path.join(config.paths.base_dir, "rules.json")

if not os.path.exists(unified_ruleset_path):
    raise FileNotFoundError("Ruleset file not found!")

with open(unified_ruleset_path, 'r') as f:
    unified_ruleset = json.load(f)

# Filter to enabled rulesets only
enabled_ruleset = {k: v for k, v in unified_ruleset.items() if v.get("enabled", True)}

print(f"  Enabled use cases: {len(enabled_ruleset)}")

Loading unified ruleset...
  Enabled use cases: 9


In [ ]:
# Run calibration from in-memory dialogues
print("\n=== Step 2: Calibrate Thresholds ===")
print("Finding optimal thresholds using Youden's J-statistic...")
print("(This may take a few minutes depending on calibration set size)")

calibration_output_dir = config.paths.calibration_dir
os.makedirs(calibration_output_dir, exist_ok=True)

optimal_params = calibrate(
    dialogue_data=calibration_dialogues,
    labels=config.labels,
    unified_ruleset=enabled_ruleset,
    output_dir=calibration_output_dir,
    show_progress=True,
    generate_plots=True,
    logger=None,
)

print("\n✓ Calibration complete!")
print(f"Optimal thresholds saved to: {calibration_output_dir}")


=== Step 2: Calibrate Thresholds ===
Finding optimal thresholds using Youden's J-statistic...
(This may take a few minutes depending on calibration set size)


Calibrating:   0%|          | 0/20 [00:00<?, ?it/s]

Calibrating: 100%|██████████| 20/20 [00:01<00:00, 10.76it/s]



✓ Calibration complete!
Optimal thresholds saved to: ../models/mistralai_Mistral-7B-Instruct-v0.2/calibration_results


### Inspect Calibration Results

Let's look at the optimal thresholds found for a few cognitive elements:

In [7]:
# Load and display optimal thresholds
thresholds_path= config.paths.thresholds_path

if os.path.exists(thresholds_path):
    with open(thresholds_path, 'r') as f:
        thresholds = json.load(f)

    print("=== Optimal Thresholds (Sample) ===")
    # Show first 5 cognitive elements
    for i, (element, params) in enumerate(list(thresholds.items())[:5]):
        threshold = params.get('threshold', 'N/A')
        patience = params.get('patience', 'N/A')
        print(f"  {element}: threshold={threshold}, patience={patience}")
    if len(thresholds) > 5:
        print(f"  ... and {len(thresholds) - 5} more")
else:
    print("Thresholds file not found. Calibration may have failed.")

=== Optimal Thresholds (Sample) ===
  buy_or_purchase: threshold=0.7, patience=1
  click_or_enter: threshold=0.05, patience=1
  content_creation: threshold=0.05, patience=1
  download_or_install: threshold=0.65, patience=1
  emotionally_engaging: threshold=0.05, patience=1
  ... and 18 more


## 6. Extract Test Dialogues (Demo Samples)

For demonstration purposes, we'll extract and process just a **few samples** from both positive and negative test sets.

**Note**: We're only processing a subset here for speed. In real evaluation, you'd process all test dialogues. You can set `max_samples_per_usecase=None` to evalute the full set.

In [8]:
# Extract test dialogues
print("=== Step 3: Extract Test Dialogues (Demo) ===")

# Set how many dialogues to extract per use case
MAX_SAMPLES_PER_USECASE = 10  # Set to None to process all dialogues from each use case
print(f"Extracting {MAX_SAMPLES_PER_USECASE} dialogues per use case in-memory...")
print(f"  Dataset: {config.paths.eval_dataset}")
print(f"  Window size: {config.eval.window_size}, stride: {config.eval.window_stride}\n")


test_dialogues = extract_dialogues_in_memory(
    dataset_root=config.paths.eval_dataset,
    splits=["positive", "negative", "neutral"],
    model=model,
    rnn_model=rnn_model,
    tokenizer=tokenizer,
    selected_layers=config.model.selected_layers,
    window_size=config.eval.window_size,
    window_stride=config.eval.window_stride,
    batch_size=32,
    max_samples_per_usecase=MAX_SAMPLES_PER_USECASE,
    logger=None,
    show_progress=True,
)

print(f"\n✓ Test dialogues extracted! ({len(test_dialogues)} dialogues)")

=== Step 3: Extract Test Dialogues (Demo) ===
Extracting 10 dialogues per use case in-memory...
  Dataset: ../data/eval
  Window size: 5, stride: 5



Extracting neutral: 100%|██████████| 2/2 [00:05<00:00,  2.67s/it]


✓ Test dialogues extracted! (200 dialogues)


### Options for Dialogue Extraction

The `extract_dialogues_in_memory` function supports:

1. **`max_samples_per_usecase`**: Limit the number of dialogues processed per use case
2. **`splits`**: Specify which splits to process (e.g., `["positive", "negative"]`)
3. **`batch_size`**: Batch size for processing (default: 32)

## 7. Evaluate on Test Samples

Now we evaluate the classifier on the extracted test logits using the calibrated thresholds. This computes:

- **TPR (True Positive Rate)**: Of all malicious dialogues, how many were correctly detected?
- **FPR (False Positive Rate)**: Of all benign dialogues, how many were incorrectly flagged?
- **AUC (Area Under Curve)**: ROC AUC and PR AUC for each use case

The evaluation uses rule-based detection:
- **Necessary conditions**: All specified cognitive elements MUST be detected.
- **Fallback condition**: Any one cognitive elements within the group MUST be detected.
- **Sufficient conditions**: These are CEs that are allowed and are optional for a given usecase.

In [ ]:
# Run evaluation on TEST samples using in-memory dialogues
print("=== Step 4: Evaluate Test Samples ===")
print("Computing TPR, FPR, and AUC metrics...")

# Use config.paths.results_dir as base, then add test subdirectory
output_dir = os.path.join(config.paths.results_dir, "test")
os.makedirs(output_dir, exist_ok=True)

results = evaluate(
    dialogue_data=test_dialogues,  # Use in-memory test dialogues
    labels=config.labels,
    thresholds_path=thresholds_path,
    unified_ruleset_path=unified_ruleset_path,
    output_dir=output_dir,
    compute_auc=True,
    logger=None,
)

print("\n✓ Evaluation complete!")
print(f"Results saved to: {output_dir}")

=== Step 4: Evaluate Test Samples ===
Computing TPR, FPR, and AUC metrics...

✓ Evaluation complete!
Results saved to: ../models/mistralai_Mistral-7B-Instruct-v0.2/results/test


## 8. Display Results

Let's examine the evaluation results:

In [10]:
# Display metrics
metrics = results["metrics"]
print("=== Evaluation Metrics ===")
print("\nMalicious Use Cases:")
print(metrics[~metrics['Usecase'].isin(['conversational', 'instructive'])].to_string(index=False))

print("\nNeutral Use Cases:")
neutral_metrics = metrics[metrics['Usecase'].isin(['conversational', 'instructive'])]
if len(neutral_metrics) > 0:
    print(neutral_metrics.to_string(index=False))
else:
    print("  (No neutral use cases in test set)")

# Display AUC if available
if "auc" in results:
    print("\n=== AUC Metrics ===")
    print(results["auc"].to_string(index=False))

# Display weighted averages
print("\n=== Weighted Averages ===")
print(results["weighted_averages"])

=== Evaluation Metrics ===

Malicious Use Cases:
                             Usecase  TPR  FPR  Accuracy       F1  Support_Pos  Support_Neg
electoral_political_content_creation  1.0  0.0      1.00 1.000000           10           10
         homophobic_content_creation  0.8  0.0      0.90 0.888889           10           10
           phishing_content_creation  1.0  0.0      1.00 1.000000           10           10
             racist_content_creation  0.8  0.1      0.85 0.842105           10           10
     reinforcing_delusional_thinking  0.8  0.0      0.90 0.888889           10           10
                     romance_baiting  1.0  0.1      0.95 0.952381           10           10
                            scamazon  0.9  0.0      0.95 0.947368           10           10
                       sql_injection  1.0  0.0      1.00 1.000000           10           10
                            tax_scam  1.0  0.0      1.00 1.000000           10           10

Neutral Use Cases:
       Usec

## Summary

This notebook demonstrated the complete evaluation pipeline:

1. ✅ **Loaded models**: Base LLM and trained RNN classifier
2. ✅ **Extracted calibration dialogues**: Processed full calibration set
3. ✅ **Calibrated thresholds**: Found optimal detection thresholds
4. ✅ **Extracted test dialogues**: Processed sample dialogues
5. ✅ **Evaluated**: Computed TPR, FPR, and AUC metrics

### Key Takeaways

- **Calibration is crucial**: Optimal thresholds significantly improve detection performance
- **Rule-based evaluation**: Uses necessary/sufficient conditions for robust detection
- **Metrics interpretation**:
  - **High TPR**: Good at catching malicious content
  - **Low FPR**: Few false alarms on benign content
  - **AUC > 0.9**: Excellent discrimination ability

### Next Steps

- Run the full pipeline from the command line: `python scripts/evaluate.py --config config.json` (uses `config.eval.max_samples_per_usecase` or full set if unset)
- For CE-level analysis and per-dialogue reports: `python scripts/evaluate_detailed.py --config config.json`
- Process full test set (set `MAX_SAMPLES_PER_USECASE = None` or use config) for comprehensive evaluation
- Tune thresholds further if needed; analyze per-use-case performance to identify weak spots

### Files Created

- `{base_dir}/calibration_results/` - Optimal thresholds and calibration plots (`thresholds.json`, plots)
- `{base_dir}/results/test/` - Evaluation metrics and visualizations (this notebook writes to a `test` subdir)

Logits are processed in-memory by default; they are not saved to disk unless `config.eval.save_logits` is enabled.